# Member 1: Synthetic data, preprocessing and state-action encoding

This notebook calls the same implementation as the CLI. It does not extract a real Sepsis-3 cohort. Run this notebook before Member 2 and Member 3.

In [1]:
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, display

root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'main_pipeline.py').is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project root or a member folder")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from main_pipeline import load_configs, prepare_data, train_model, evaluate_model

output = Path(os.environ.get('ASS2_OUTPUT_DIR', root)).resolve()
print("Source: synthetic coursework demo; no clinical efficacy claim.")
print("Output:", output)


Source: synthetic coursework demo; no clinical efficacy claim.
Output: C:\Users\ADMIN\Desktop\ass2


## 1. Prepare the reproducible cohort

Existing outputs are reused only when configuration matches. To regenerate, use the CLI with an explicit output directory and `--overwrite`. Current treatment effects enter the next state, not the current observation.

In [2]:
configs = load_configs()
manifest_path = output / 'artifacts/run_config.json'
if (output / 'data/raw_sepsis_cohort.csv').exists():
    if not manifest_path.exists() or json.loads(manifest_path.read_text(encoding='utf-8')) != configs:
        raise RuntimeError("Existing run has a different or missing configuration; select a new output directory")
    print("Reusing prepared synthetic data.")
else:
    prepare_data(output, configs)
raw = pd.read_csv(output / 'data/raw_sepsis_cohort.csv')
print("Observations:", len(raw), "| Patients:", raw['subject_id'].nunique())
raw.head()


Reusing prepared synthetic data.
Observations: 7840 | Patients: 600


,subject_id,hadm_id,icustay_id,step,age,gender,elixhauser,sofa_initial,gcs,hr,...,bicarbonate,ph,pao2,paco2,hemoglobin,urine_output,cumulated_balance,iv_fluids,vasopressors,died
0,10000,150000,200000,0,70,1,2,4,14,85.281478,...,26.953814,7.345519,85.217239,32.982073,11.463943,106.471457,86.618209,0.000000,0.0,0
1,10000,150000,200000,1,70,1,2,4,14,82.894585,...,27.595503,7.326843,88.354072,32.059760,11.530521,78.857117,-19.853248,211.265613,0.0,0
2,10000,150000,200000,2,70,1,2,4,14,82.052167,...,28.045974,7.320291,84.907680,31.672483,11.622713,51.869317,112.555248,0.000000,0.0,0
3,10000,150000,200000,3,70,1,2,4,14,78.994948,...,28.252916,7.318689,83.942159,33.822756,11.835871,1.894134,60.685931,274.549305,0.0,0
4,10000,150000,200000,4,70,1,2,4,14,83.435934,...,28.910833,7.327249,74.063364,34.545724,12.021163,0.000000,333.341102,347.504654,0.0,0


## 2. Patient-level splits and episode boundaries

All stays belonging to a patient remain in one split; each ICU stay is a separate episode.

In [3]:
trajectories = {name: pd.read_csv(output / f'data/trajectories_{name}.csv') for name in ['train', 'val', 'test']}
ids = {name: set(frame['subject_id']) for name, frame in trajectories.items()}
assert not (ids['train'] & ids['val'] or ids['train'] & ids['test'] or ids['val'] & ids['test'])
pd.DataFrame({name: {'patients': len(ids[name]), 'episodes': int(frame['done'].sum()), 'steps': len(frame)} for name, frame in trajectories.items()}).T


,patients,episodes,steps
train,420,420,5496
val,90,90,1173
test,90,90,1171


## 3. Train-fitted preprocessing and encoding

Training medians fill missing values; the StandardScaler and K-means centroids fit training observations only. Positive dose quartiles also come from training.

In [4]:
preprocessor = joblib.load(output / 'artifacts/preprocessor.joblib')
encoder = joblib.load(output / 'artifacts/state_action_encoder.joblib')
print("Feature count:", len(preprocessor.feature_cols))
print("Health states:", encoder.n_states, "| Terminal states:", encoder.alive_state, encoder.dead_state)
print("Fluid cutoffs:", encoder.fluid_bins)
print("Vasopressor cutoffs:", encoder.vaso_bins)
pd.Series(preprocessor.medians, name='training_median')


Feature count: 30
Health states: 75 | Terminal states: 75 76
Fluid cutoffs: [0.0, 124.14998776989034, 221.728807284115, 389.49880625947645]
Vasopressor cutoffs: [0.0, 0.07168748527282387, 0.13672317313743432, 0.2715441008627684]


age                   64.000000
gender                 0.000000
elixhauser             3.000000
sofa_initial           6.000000
gcs                   15.000000
hr                    87.151563
sys_bp               128.349449
dia_bp                80.604580
map                   96.277624
rr                    20.357591
temp                  37.533599
spo2                  95.747414
fio2                   0.400000
lactate                2.229489
creatinine             1.585166
platelets            224.601092
bilirubin              1.165217
wbc                   12.193611
bun                   25.364373
glucose              136.650984
potassium              4.145794
sodium               137.998306
chloride             102.195814
bicarbonate           21.618776
ph                     7.334584
pao2                  95.799733
paco2                 40.444159
hemoglobin            10.440103
urine_output         180.134894
cumulated_balance    544.824728
Name: training_median, dtype: float64

## 4. Inspect the trajectory contract

Intermediate rewards are zero; the final transition receives +100 or -100 from a simulated outcome. These are not measured 90-day outcomes.

In [5]:
first_subject = trajectories['train']['subject_id'].iloc[0]
trajectories['train'].query('subject_id == @first_subject')


,subject_id,hadm_id,icustay_id,time_step,state,action,iv_fluids,vasopressors,next_state,reward,done,outcome_type,dataset_split,died
0,10000,150000,200000,0,36,0,0.000000,0.000000,36,0.0,False,survived,train,0
1,10000,150000,200000,1,36,10,211.265613,0.000000,36,0.0,False,survived,train,0
2,10000,150000,200000,2,36,0,0.000000,0.000000,36,0.0,False,survived,train,0
3,10000,150000,200000,3,36,15,274.549305,0.000000,36,0.0,False,survived,train,0
4,10000,150000,200000,4,36,15,347.504654,0.000000,36,0.0,False,survived,train,0
5,10000,150000,200000,5,36,0,0.000000,0.000000,36,0.0,False,survived,train,0
6,10000,150000,200000,6,36,22,394.302709,0.092288,36,0.0,False,survived,train,0
7,10000,150000,200000,7,36,5,57.001321,0.000000,36,0.0,False,survived,train,0
8,10000,150000,200000,8,36,0,0.000000,0.000000,75,100.0,True,survived,train,0


## 5. Audit and provenance

The audit reports checks it actually performed. The raw-data report cannot independently prove model fit provenance.

In [6]:
display(json.loads((output / 'artifacts/cohort_manifest.json').read_text()))
from IPython.display import Markdown
display(Markdown((output / 'docs/data_quality_report.md').read_text(encoding='utf-8')))


{'cohort_version': 'synthetic-v2',
 'data_source': 'synthetic',
 'clinical_validation': False,
 'generation_parameters': {'dataset_name': 'Synthetic sepsis coursework demo',
  'version': 'synthetic-v2',
  'n_patients': 600,
  'time_step_hours': 4,
  'max_window_hours': 72,
  'train_ratio': 0.7,
  'val_ratio': 0.15,
  'test_ratio': 0.15,
  'random_state': 42,
  'min_age': 18,
  'sepsis_criteria': 'Not clinically validated; simulated adult ICU-like observations',
  'imputation_strategy': 'median',
  'scaling_strategy': 'standard_scaler',
  'features': ['age',
   'gender',
   'elixhauser',
   'sofa_initial',
   'gcs',
   'hr',
   'sys_bp',
   'dia_bp',
   'map',
   'rr',
   'temp',
   'spo2',
   'fio2',
   'lactate',
   'creatinine',
   'platelets',
   'bilirubin',
   'wbc',
   'bun',
   'glucose',
   'potassium',
   'sodium',
   'chloride',
   'bicarbonate',
   'ph',
   'pao2',
   'paco2',
   'hemoglobin',
   'urine_output',
   'cumulated_balance']},
 'cohort_definition': 'Simulated adul

# Data Quality and Preprocessing Audit Report

## 1. Cohort Overview
- Total ICU Observations: 7840
- Unique patients: 600
- Source: synthetic coursework demo. This audit does not verify Sepsis-3 eligibility.
- Average Steps per Patient: 13.1
- Crude Mortality Rate: 4.83%

## 2. Feature Distribution and Missingness Summary

| Feature | Missing Count | Missing % | Min | Median | Max | Outliers (Tukey) |
| --- | --- | --- | --- | --- | --- | --- |
| age | 0 | 0.0% | 27.00 | 64.00 | 90.00 | 97 |
| gender | 0 | 0.0% | 0.00 | 1.00 | 1.00 | 0 |
| elixhauser | 0 | 0.0% | 0.00 | 3.00 | 10.00 | 97 |
| sofa_initial | 0 | 0.0% | 2.00 | 6.00 | 16.00 | 12 |
| gcs | 0 | 0.0% | 8.00 | 15.00 | 15.00 | 1010 |
| hr | 0 | 0.0% | 45.00 | 87.17 | 135.23 | 36 |
| sys_bp | 0 | 0.0% | 55.00 | 128.31 | 190.00 | 6 |
| dia_bp | 0 | 0.0% | 30.00 | 80.03 | 110.00 | 0 |
| map | 0 | 0.0% | 52.67 | 96.10 | 136.67 | 0 |
| rr | 0 | 0.0% | 8.00 | 19.98 | 42.00 | 66 |
| temp | 0 | 0.0% | 35.00 | 37.52 | 40.50 | 27 |
| spo2 | 0 | 0.0% | 77.83 | 95.83 | 100.00 | 114 |
| fio2 | 0 | 0.0% | 0.21 | 0.40 | 0.80 | 0 |
| lactate | 0 | 0.0% | 0.50 | 2.25 | 12.36 | 325 |
| creatinine | 0 | 0.0% | 0.30 | 1.53 | 9.44 | 355 |
| platelets | 0 | 0.0% | 15.00 | 223.51 | 414.18 | 79 |
| bilirubin | 0 | 0.0% | 0.10 | 1.18 | 6.12 | 363 |
| wbc | 0 | 0.0% | 1.00 | 12.51 | 32.01 | 34 |
| bun | 0 | 0.0% | 5.00 | 24.59 | 54.54 | 1 |
| glucose | 0 | 0.0% | 40.00 | 135.72 | 257.28 | 20 |
| potassium | 0 | 0.0% | 2.41 | 4.16 | 5.80 | 72 |
| sodium | 0 | 0.0% | 123.88 | 138.05 | 154.97 | 80 |
| chloride | 0 | 0.0% | 84.04 | 102.06 | 120.86 | 20 |
| bicarbonate | 0 | 0.0% | 10.00 | 21.67 | 35.96 | 91 |
| ph | 0 | 0.0% | 7.08 | 7.34 | 7.60 | 29 |
| pao2 | 0 | 0.0% | 45.00 | 95.57 | 161.74 | 24 |
| paco2 | 0 | 0.0% | 18.13 | 40.60 | 66.94 | 38 |
| hemoglobin | 0 | 0.0% | 4.50 | 10.39 | 18.00 | 102 |
| urine_output | 0 | 0.0% | 0.00 | 180.85 | 470.97 | 58 |
| cumulated_balance | 0 | 0.0% | -3305.88 | 549.89 | 5135.77 | 361 |
| iv_fluids | 0 | 0.0% | 0.00 | 120.07 | 2000.00 | 349 |
| vasopressors | 0 | 0.0% | 0.00 | 0.00 | 1.29 | 1238 |

## 3. Data Leakage and Integrity Verification
- Temporal ordering and contiguous steps: PASS
- Consistent binary outcome per episode: PASS
- Patient-disjoint splits covering raw cohort: PASS
- Model fit provenance is not verifiable from raw CSV; inspect pipeline fit calls and regression tests.